# CALIMA Dust Chemistry Solver — Tutorial Notebook

This notebook walks through the full CALIMA workflow:

1. **JSON configuration** — understanding every key field  
2. **Dust process flags** — toggling physics on/off  
3. **Single RK4 run** — time evolution for one (T, nH) point  
4. **Single equilibrium run** — comparing NK steady-state with the RK4 final state  
5. **T–nH grid** — running the RK4 solver for 5 Myr over a 2-D parameter grid  
6. **Grid plots** — DTM, PAH abundance, small/large grain fractions as 2-D heatmaps


## 1. Import Required Libraries


In [ ]:
import json
import copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import LogLocator, LogFormatter

# CALIMA path helpers -- resolve bundled configs and generated tables
# independently of the current working directory.
from pycalima._paths import (
    get_model_data_dir,
    get_results_dir,
    resolve_solver_config_path,
)

# CALIMA solver modules
from pycalima.solvers.run_chemistry import run_chemistry, compute_element_totals
from pycalima.solvers.run_grid import run_grid, save_grid_npz
from pycalima.solvers.dust_init import load_initial_conditions, SEC2MYR
from pycalima.solvers.rhs import build_process_list
from pycalima.solvers.plotting import plot_chemistry_evolution

# Plotting defaults
plt.rcParams.update({
    "figure.dpi":    120,
    "axes.grid":     True,
    "grid.alpha":    0.3,
    "font.size":     11,
})

# Where generated tables are read from and notebook output is written to.
# Both are CWD-independent; MODEL_DATA honours $CALIMA_MODEL_DATA.
MODEL_DATA = get_model_data_dir()
OUT = get_results_dir("notebooks")
print(f"Generated tables : {MODEL_DATA}")
print(f"Notebook output  : {OUT}")


---
## 2. Understanding the JSON Configuration File

Every CALIMA run is driven by a single JSON file. It has five top-level sections:

| Section | Purpose |
|---|---|
| `environment` | Gas temperature, density, UV field, molecular weight |
| `elemental_abundances` | Solar (or custom) mass fractions for H, He, C, N, O, Mg, Si, S, Fe |
| `dust_bins` | One entry per size bin — composition, grain size, initial density, interaction flags |
| `pah_bins` | One entry per PAH family — number of C atoms, cluster flag, initial density |
| `physics` | Boolean flags to enable/disable each process |
| `models` | Model variant strings (coagulation kernel, shattering, photolysis, …) |
| `solver` | Solver type (`rk4`, `newton_krylov`, `sparse_newton`), end time, step-size controls |

Let's load the example config and inspect it:


In [ ]:
CFG_PATH = resolve_solver_config_path("example_ic")

with CFG_PATH.open() as fh:
    cfg = json.load(fh)

# Pretty-print the full config
print(json.dumps(cfg, indent=2))


In [ ]:
# --- Anatomy of the key fields ---

print("=== environment ===")
for k, v in cfg["environment"].items():
    print(f"  {k:40s} = {v}")

print("\n=== dust bins (first bin only) ===")
b0 = cfg["dust_bins"][0]
for k, v in b0.items():
    print(f"  {k:40s} = {v}")

print("\n=== pah bins ===")
for pb in cfg["pah_bins"]:
    print(f"  {pb['id']:15s}  nc={pb['nc']:3d}  cluster={pb['is_cluster']}  "
          f"rho_init={pb['initial_mass_density_gcm3']:.1e} g/cm3")

print("\n=== physics flags ===")
for k, v in cfg["physics"].items():
    status = "ON " if v else "off"
    print(f"  {status}  {k}")

print("\n=== solver ===")
for k, v in cfg["solver"].items():
    print(f"  {k:25s} = {v}")


---
## 3. Setting Up Dust Processes and Initial Conditions

### 3a. Toggling physics flags

The `physics` block is the main switch panel. Each flag turns a full rate module on or off — you can mix and match freely:


In [ ]:
import tempfile, os

def make_config(
    base_cfg: dict,
    *,
    T_K:   float | None = None,
    nH:    float | None = None,
    G0:    float | None = None,
    t_end_Myr: float | None = None,
    physics_overrides: dict | None = None,
    solver_type: str | None = None,
    f_tol: float | None = None,
) -> Path:
    """
    Clone a base config dict, apply overrides, write to a temp file,
    and return its Path.  Caller is responsible for unlinking when done.
    """
    c = copy.deepcopy(base_cfg)

    if T_K  is not None: c["environment"]["gas_temperature_K"]           = T_K
    if nH   is not None: c["environment"]["hydrogen_number_density_cm3"] = nH
    if G0   is not None: c["environment"]["radiation_field_G0"]          = G0
    if t_end_Myr is not None: c["solver"]["t_end_Myr"]                   = t_end_Myr
    if solver_type is not None: c["solver"]["type"]                      = solver_type
    if f_tol is not None: c["solver"]["f_tol"]                           = f_tol

    if physics_overrides:
        c["physics"].update(physics_overrides)

    fd, tmp_path = tempfile.mkstemp(suffix=".json", prefix="calima_")
    os.close(fd)
    with open(tmp_path, "w") as fh:
        json.dump(c, fh, indent=2)
    return Path(tmp_path)


# Demo: print active processes for the "all-on" variant
all_on = copy.deepcopy(cfg)
for k in all_on["physics"]:
    all_on["physics"][k] = True

p = make_config(all_on)
state, _, _ = load_initial_conditions(p)
processes = build_process_list(state)
print("Active processes (all-on):")
for proc in processes:
    print(f"  • {proc.name}")
os.unlink(p)


### 3b. Inspecting initial conditions

The `dust_bins` and `pah_bins` blocks set the **initial mass density** of each grain family.  
The `elemental_abundances` block sets the total element budget (gas + dust + PAH).  
The difference between total and dust-locked mass is the initial gas-phase reservoir.


In [ ]:
state, y_gas_0, y_dust_0 = load_initial_conditions(CFG_PATH)
el_total = compute_element_totals(state, y_gas_0, y_dust_0)

print(f"{'Element':>6}  {'Total [g/cm3]':>14}  {'Gas [g/cm3]':>14}  "
      f"{'Dust+PAH [g/cm3]':>18}  {'Depletion':>10}")
print("-" * 72)
for i, name in enumerate(state.el_names):
    tot  = el_total[i]
    gas  = y_gas_0[i]
    dust = tot - gas
    depl = dust / tot * 100 if tot > 0 else 0.0
    print(f"{name:>6}  {tot:>14.4e}  {gas:>14.4e}  {dust:>18.4e}  {depl:>9.2f}%")

print()
print(f"\nDust bins:  {state.ndust}   PAH bins: {state.npah}")
print(f"Total rho_dust = {y_dust_0.sum():.4e} g/cm3")
print(f"Total rho_gas  = {y_gas_0.sum():.4e} g/cm3")
print(f"DTM (initial)  = {y_dust_0.sum() / el_total[[i for i,n in enumerate(state.el_names) if n in ('C','O','Mg','Si','Fe')]].sum():.4f}")


---
## 4. Running a Single RK4 Solver Test

We run the adaptive RK4 integrator for **5 Myr** starting from the CNM example conditions  
(T = 100 K, nH = 100 cm⁻³, G0 = 1).  All standard processes are on.


In [ ]:
# Build a config for the RK4 single-point run
# All physics on, t_end = 5 Myr, RK4 solver
rk4_single_cfg = copy.deepcopy(cfg)
for k in rk4_single_cfg["physics"]:
    rk4_single_cfg["physics"][k] = True
rk4_single_cfg["solver"]["type"]       = "rk4"
rk4_single_cfg["solver"]["t_end_Myr"]  = 5.0
rk4_single_cfg["solver"]["countmax"]   = 50000

tmp_rk4 = make_config(rk4_single_cfg, T_K=100.0, nH=100.0, G0=1.0)

rk4_results = run_chemistry(tmp_rk4, verbose=True)
os.unlink(tmp_rk4)

print(f"\nWall time : {rk4_results['elapsed_s']:.2f} s")
print(f"Steps     : {rk4_results['diagnostics']['naccepted']} accepted, "
      f"{rk4_results['diagnostics']['nrejected']} rejected")


In [ ]:
## --- Plot time evolution ---

hist    = rk4_results["diagnostics"]["history"]
t_Myr   = hist["time_s"] / SEC2MYR
y_gas_h = hist["y_gas"]    # shape (nsnaps, n_el)
y_dust_h= hist["y_dust"]   # shape (nsnaps, npah+ndust)

state   = rk4_results["state"]
npah    = state.npah
el_names = state.el_names
dust_ids = [db.bin_id for db in state.dust_bins]
pah_ids  = [pb.bin_id for pb in state.pah_bins]

# Compute DTM at every snapshot
el_tot_h = np.array([
    compute_element_totals(state, y_gas_h[k], y_dust_h[k])
    for k in range(len(t_Myr))
])
# Dust-forming elements: C, O, Mg, Si, Fe
dust_el_idx = [el_names.index(e) for e in ("C", "O", "Mg", "Si", "Fe") if e in el_names]
rho_dust_total = y_dust_h.sum(axis=1)
rho_metals     = el_tot_h[:, dust_el_idx].sum(axis=1)
DTM_h          = np.where(rho_metals > 0, rho_dust_total / rho_metals, 0.0)
rho_pah_total  = y_dust_h[:, :npah].sum(axis=1)

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

# Panel 1 — dust grain bins
ax = axes[0]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, state.ndust))
for i, (did, c) in enumerate(zip(dust_ids, colors)):
    ax.semilogy(t_Myr, y_dust_h[:, npah + i], label=did, color=c, lw=1.8)
ax.set_ylabel(r"$\rho_{\rm dust}$  [g cm$^{-3}$]")
ax.set_title(r"Dust grain bins  ($T=100\,\mathrm{K}$, $n_H=100\,\mathrm{cm}^{-3}$, $G_0=1$, 5 Myr)")
ax.legend(fontsize=9, ncol=2)

# Panel 2 — PAH bins + DTM
ax = axes[1]
colors_pah = plt.cm.cool(np.linspace(0.2, 0.8, npah))
for i, (pid, c) in enumerate(zip(pah_ids, colors_pah)):
    ax.semilogy(t_Myr, np.maximum(y_dust_h[:, i], 1e-50), label=pid, color=c, lw=1.8)
ax.set_ylabel(r"$\rho_{\rm PAH}$  [g cm$^{-3}$]")
ax.legend(fontsize=9)

# Panel 3 — DTM ratio
ax = axes[2]
ax.plot(t_Myr, DTM_h, color="steelblue", lw=2)
ax.set_xlabel("Time  [Myr]")
ax.set_ylabel("DTM  (dust / metals)")
ax.set_ylim(bottom=0)

fig.tight_layout()


---
## 5. Running a Single Equilibrium Solver Test

The `newton_krylov` solver finds the **steady-state** (dρ/dt = 0) in one shot — no time-stepping.  
We use the same starting conditions as the RK4 run above and compare the final states.

> **Note:** The equilibrium solver finds the mathematical fixed point of the rate equations.  
> It does **not** correspond to the RK4 solution at *t* = 5 Myr unless that run has truly converged — a useful diagnostic for whether 5 Myr is long enough.


In [ ]:
EQ_CFG_PATH = resolve_solver_config_path("equilibrium_postshock_test")

eq_results = run_chemistry(EQ_CFG_PATH, verbose=True)

diag = eq_results["diagnostics"]
F0    = diag.get("F0_norm",    "N/A")
Ffin  = diag.get("F_final_norm","N/A")

def _fmt(v):
    return f"{v:.3e}" if isinstance(v, (int, float)) else str(v)

print(f"\nConverged  : {diag.get('converged', 'N/A')}")
print(f"||F_init|| : {_fmt(F0)}")
print(f"||F_final||: {_fmt(Ffin)}")
print(f"Iterations : {diag.get('nfev', 'N/A')}")
print(f"Solver     : {diag.get('solver_name', 'N/A')}")


In [ ]:
## --- Compare RK4 final state vs equilibrium ---

rk4_dust_final = rk4_results["y_dust_final"]
eq_dust_final  = eq_results["y_dust_final"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

nbins     = state.ndust + npah
labels    = pah_ids + dust_ids
x_pos     = np.arange(nbins)
bar_width  = 0.38

# RK4 final — left bars
ax1.bar(x_pos - bar_width/2, np.maximum(rk4_dust_final, 1e-50), bar_width,
        label="RK4 @5 Myr", color="steelblue", alpha=0.85)
ax1.bar(x_pos + bar_width/2, np.maximum(eq_dust_final,  1e-50), bar_width,
        label="Equilibrium (NK)", color="tomato", alpha=0.85)
ax1.set_yscale("log")
ax1.set_xticks(x_pos)
ax1.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax1.set_ylabel(r"$\rho$  [g cm$^{-3}$]")
ax1.set_title("Dust+PAH densities: RK4 vs Equilibrium")
ax1.legend()

# Fractional difference
# np.divide with `where=` rather than np.where: the latter evaluates both
# branches, so it still divides by the zero entries and warns.
rel_diff = np.divide(
    rk4_dust_final - eq_dust_final,
    eq_dust_final,
    out=np.zeros_like(eq_dust_final, dtype=float),
    where=eq_dust_final > 1e-50,
)
ax2.bar(x_pos, rel_diff, color=["steelblue" if d >= 0 else "tomato" for d in rel_diff])
ax2.axhline(0, color="k", lw=0.8)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax2.set_ylabel(r"$(\rho_{\rm RK4} - \rho_{\rm eq}) / \rho_{\rm eq}$")
ax2.set_title("Fractional difference  (RK4 @5 Myr  −  Equilibrium) / Equilibrium")

fig.tight_layout()


---
## 6. Building the Temperature–Density Grid

We define logarithmically spaced arrays in $T$ and $n_H$ (or $G_0$), then construct a 2-D mesh.  
This is passed directly to `run_grid()` — no explicit `meshgrid` loop needed.


In [ ]:
# ---- T–nH grid parameters ----
T_min,  T_max  =   50.0,  20000.0    # Kelvin
nH_min, nH_max =    0.1,   1000.0    # cm-3

n_T  = 8    # number of temperature points
n_nH = 6    # number of density points

T_values  = np.logspace(np.log10(T_min),  np.log10(T_max),  n_T)
nH_values = np.logspace(np.log10(nH_min), np.log10(nH_max), n_nH)

# Visualise the grid layout
T_grid, nH_grid = np.meshgrid(T_values, nH_values, indexing="ij")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(T_grid.ravel(), nH_grid.ravel(), marker="s", s=60,
           c=np.log10(T_grid.ravel()), cmap="plasma", zorder=3)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"Temperature $T$  [K]")
ax.set_ylabel(r"Hydrogen density $n_H$  [cm$^{-3}$]")
ax.set_title(f"T–$n_H$ grid  ({n_T} × {n_nH} = {n_T*n_nH} points)")
fig.tight_layout()
# ---- optional T–G0 variant ----
G0_values  = np.logspace(-1, 2, 6)     # 0.1 … 100 G0
T_G0_grid, G0_grid = np.meshgrid(T_values, G0_values, indexing="ij")
print(f"T–G0 grid would be  {n_T} × {len(G0_values)} = {n_T*len(G0_values)} points")
print(f"  T  values : {np.round(T_values,  1)}")
print(f"  nH values : {np.round(nH_values, 3)}")
print(f"  G0 values : {np.round(G0_values, 3)}")


---
## 7. Running the RK4 Solver Over the T-nH Grid

`run_grid()` distributes all (T, nH) pairs across CPU cores using **joblib with the `loky` backend**
(process-based, not thread-based — each worker runs in a separate Python interpreter so there is no GIL contention).

- `n_jobs=-1` uses every logical core on the machine.
- `n_jobs=N` uses exactly N cores.
- `n_jobs=1` runs serially (no joblib dependency needed).

The base JSON config is written once to a temporary file; every worker reads it independently
(read-only, so there are no race conditions).
Results are collected into 2-D numpy arrays indexed `[i_T, i_nH]`.

> **Estimated runtime:** the 8x6 = 48-point grid takes roughly 1-5 min of CPU time total,
> so on an 8-core machine you should see wall-clock times of 10-40 s.


In [ ]:
import os

# Ensure joblib is available for process-based parallelism
try:
    import joblib as _jl
    _n_cpus = _jl.cpu_count()
    print(f"joblib {_jl.__version__}  --  {_n_cpus} logical CPU cores detected")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "joblib"])
    import joblib as _jl
    _n_cpus = _jl.cpu_count()
    print(f"joblib installed  --  {_n_cpus} logical CPU cores detected")

# Base config: all physics on, G0 = 1 (standard ISM), RK4 for 5 Myr
all_on_cfg = copy.deepcopy(cfg)
for k in all_on_cfg["physics"]:
    all_on_cfg["physics"][k] = True
all_on_cfg["solver"]["type"]      = "rk4"
all_on_cfg["solver"]["t_end_Myr"] = 5.0
all_on_cfg["solver"]["countmax"]  = 50000

# Write the base config once; all worker processes read it (never write it)
tmp_all = make_config(all_on_cfg)

grid = run_grid(
    config_path = tmp_all,
    x_param     = "T",
    x_values    = T_values,
    y_param     = "nH",
    y_values    = nH_values,
    t_end_Myr   = 5.0,
    solver_type = "rk4",
    # n_jobs=-1  -> use all CPU cores (joblib loky, process-based, no GIL contention)
    # n_jobs= N  -> use exactly N cores
    # n_jobs= 1  -> serial fallback
    n_jobs      = -1,
    verbose     = True,
)
os.unlink(tmp_all)

NPZ_PATH = OUT / "grid_T_nH_5Myr.npz"
save_grid_npz(grid, NPZ_PATH)
total_cpu = grid["elapsed_s"].sum()
print(f"\nGrid saved      -> {NPZ_PATH}")
print(f"DTM array shape : {grid['DTM'].shape}  (n_T={n_T}, n_nH={n_nH})")
print(f"Converged       : {grid['converged'].sum()} / {n_T * n_nH} points")
print(f"Total CPU time  : {total_cpu:.1f} s  (wall time approx {total_cpu/_n_cpus:.1f} s with {_n_cpus} cores)")


---
## 8. Plotting DTM, PAH and Dust Abundances Across the Grid

We plot four quantities on log T vs log nH heatmaps:
1. **DTM** — dust-to-metal mass ratio  
2. **PAH abundance** — total PAH mass density [g cm⁻³]  
3. **Small-grain fraction** — mass fraction in the smallest dust bin  
4. **Large-grain fraction** — mass fraction in the largest dust bin  


In [ ]:
from pycalima.solvers.run_grid import load_grid_npz

grid = load_grid_npz(NPZ_PATH)

T_ax  = np.array(grid["x_values"])   # (n_T,)
nH_ax = np.array(grid["y_values"])   # (n_nH,)

# Grid arrays: shape (n_T, n_nH)
DTM       = np.array(grid["DTM"])
rho_pah   = np.array(grid["rho_pah"]).sum(axis=2)   # sum over PAH bins
rho_dust  = np.array(grid["rho_dust"])               # (n_T, n_nH, n_dust)

# Small = first dust bin, large = last dust bin
rho_dust_total = rho_dust.sum(axis=2)
small_frac = np.where(rho_dust_total > 0, rho_dust[:, :, 0]  / rho_dust_total, 0.0)
large_frac = np.where(rho_dust_total > 0, rho_dust[:, :, -1] / rho_dust_total, 0.0)

# Mask unconverged points
converged = np.array(grid["converged"])
mask      = ~converged

# ---- plotting helper ----
def _pcolormesh_logaxes(ax, T, nH, Z, label, cmap, norm=None, mask=None):
    # pcolormesh with log x and y axes
    # Build edges for pcolormesh
    def log_edges(v):
        lv  = np.log10(v)
        dlv = np.diff(lv)
        edges = np.concatenate([[lv[0] - dlv[0]/2],
                                 lv[:-1] + dlv/2,
                                 [lv[-1] + dlv[-1]/2]])
        return 10**edges
    Te  = log_edges(T)
    nHe = log_edges(nH)
    Tg, nHg = np.meshgrid(Te, nHe, indexing="ij")

    Zp = Z.copy().astype(float)
    if mask is not None:
        Zp[mask] = np.nan
    pcm = ax.pcolormesh(Tg, nHg, Zp, cmap=cmap, norm=norm, shading="flat")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"$T$  [K]"); ax.set_ylabel(r"$n_H$  [cm$^{-3}$]")
    plt.colorbar(pcm, ax=ax, label=label, pad=0.02)
    return pcm


fig, axes = plt.subplots(2, 2, figsize=(13, 10))

_pcolormesh_logaxes(axes[0, 0], T_ax, nH_ax, DTM,
                    label="DTM",
                    cmap="viridis",
                    norm=mcolors.LogNorm(vmin=max(DTM[converged].min(), 1e-4),
                                        vmax=DTM[converged].max()),
                    mask=mask)
axes[0, 0].set_title("Dust-to-Metal ratio  (5 Myr, RK4)")

_pcolormesh_logaxes(axes[0, 1], T_ax, nH_ax, np.maximum(rho_pah, 1e-50),
                    label=r"$\rho_{\rm PAH}$  [g cm$^{-3}$]",
                    cmap="magma",
                    norm=mcolors.LogNorm(vmin=1e-32, vmax=max(rho_pah[converged].max(), 1e-32)),
                    mask=mask)
axes[0, 1].set_title("Total PAH mass density  (5 Myr, RK4)")

_pcolormesh_logaxes(axes[1, 0], T_ax, nH_ax, small_frac,
                    label="Small-grain mass fraction",
                    cmap="cividis",
                    mask=mask)
axes[1, 0].set_title(f"Small-grain fraction  [{grid['dust_bin_ids'][0]}]")

_pcolormesh_logaxes(axes[1, 1], T_ax, nH_ax, large_frac,
                    label="Large-grain mass fraction",
                    cmap="inferno",
                    mask=mask)
axes[1, 1].set_title(f"Large-grain fraction  [{grid['dust_bin_ids'][-1]}]")

fig.suptitle(r"$T$–$n_H$ grid  (all processes on,  $G_0=1$,  5 Myr RK4)", fontsize=13, y=1.01)
fig.tight_layout()


In [ ]:
## --- Optional: elemental depletion across the grid ---

el_names = list(grid["el_names"])
rho_gas  = np.array(grid["rho_gas"])   # (n_T, n_nH, n_el)

# Depletion of Si as a representative refractory element
if "Si" in el_names:
    i_Si = el_names.index("Si")
    depl_Si = np.array(grid["depletions"])[:, :, i_Si]

    fig, ax = plt.subplots(figsize=(7, 5))
    _pcolormesh_logaxes(ax, T_ax, nH_ax, depl_Si * 100,
                        label="Si depletion  [%]",
                        cmap="RdYlGn_r",
                        mask=mask)
    ax.set_title(r"Silicon depletion into dust  (5 Myr, RK4)")
    fig.tight_layout()


---
## 9. Dust Thermal Sublimation

Dust grains exposed to intense radiation fields can reach temperatures high enough to
sublimate.  CALIMA implements the **Guhathakurta & Draine (1989, ApJ 345, 230)** (GD89)
formalism, which uses the RRK micro-canonical correction factor to the vapour-pressure
relation of Draine & Salpeter (1979).

For **small grains** (≲ 100 Å), single photon absorption causes large temperature spikes,
so the instantaneous grain temperature is stochastic.  CALIMA uses the full transition-matrix
method from **Camps et al. (2015, A&A 580, A87)** (the SKIRT algorithm) to compute the
equilibrium temperature probability distribution $P(T)$, and then averages the sublimation
rate over this distribution:

$$\langle \varepsilon \rangle = \int \varepsilon(T)\, P(T)\, \mathrm{d}T, \qquad \varepsilon(T) \equiv \frac{1}{a}\left|\frac{\mathrm{d}a}{\mathrm{d}t}\right|_{\rm GD89}$$

The results are pre-tabulated in `model_data/dust_sublimation/erosion_rate_{bin_id}.dat`
and loaded at solver start-up.  In the ODE the fractional mass-loss rate is

$$\frac{1}{\rho}\frac{\mathrm{d}\rho}{\mathrm{d}t} = -3\,\langle\varepsilon\rangle$$

because grain mass scales as $a^3$.

The following cells demonstrate the sublimation diagnostics available in
`models.dust_radiation.dust_sublimation`.


In [ ]:
from pycalima.models.dust_radiation.dust_sublimation import (
    plot_sublimation,
    plot_temperature_distribution,
    plot_sublimation_rate_vs_temperature,
    write_sublimation_rate_tables,
    adaptive_temperature_distribution,
)

# Verify that the module imports cleanly
print("dust_sublimation module loaded successfully")


### 9a. Sublimation timescale vs. radiation field

`plot_sublimation` shows how the effective sublimation timescale (or |da/dt|) varies
with the Habing radiation field strength G₀ and the distance from the source for each
non-PAH dust bin.  Large grains are treated in thermal equilibrium; small grains use
the stochastic temperature distribution.


In [ ]:
# Sublimation timescale vs G0 (Mathis ISRF) and vs distance from an O6V star.
plot_sublimation(
    G0_min=1.0, G0_max=1e7,
    n_G0=200,
    quantity='timescale',
    show=True,
)


### 9b. Stochastic temperature distributions

`plot_temperature_distribution` shows $P(T)$ for each dust bin at three representative
radiation-field strengths.  Large grains that satisfy the equilibrium criterion
(ΔT < 10 K) are displayed as a vertical marker at their equilibrium temperature.


In [ ]:
# Stochastic P(T) at three radiation-field intensities.
plot_temperature_distribution(
    G0_values=[1e1, 1e3, 1e5],
    n_bins=300,
    show=True,
)


### 9c. Erosion rate ε(T) for each dust bin

`plot_sublimation_rate_vs_temperature` shows the specific erosion rate
ε = |da/dt|/a [s⁻¹] as a function of temperature for all dust bins in a
single panel, with reference lines at 1 yr, 10³ yr, 10⁶ yr, and the age
of the Universe.


In [ ]:
# Erosion rate ε(T) for all dust bins.
plot_sublimation_rate_vs_temperature(
    T_min=50.0, T_max=3000.0,
    n_T=500,
    show=True,
)


### 9d. Pre-computing and saving the sublimation-rate tables

The CALIMA ODE solver reads pre-tabulated sublimation rates at start-up
(two-column ASCII files readable by Fortran).  To regenerate the tables
(e.g. after changing the radiation field model or the grain-size config),
call `write_sublimation_rate_tables`.  Tables are written to
`model_data/dust_sublimation/sublimation_rate_{bin_id}.dat`.


In [ ]:
# (Re-)generate the sublimation-rate tables for all non-PAH dust bins.
# These files are loaded automatically by dust_init.py at solver start-up.
paths = write_sublimation_rate_tables(
    T_min=50.0, T_max=3000.0,
    n_T=300,
)
print(f"Sublimation-rate tables written ({len(paths)} files):")
for p in paths:
    print(f"  {p}")


### 9e. Enabling sublimation in the ODE solver

To include sublimation in a CALIMA chemistry run, add `"dust_sublimation": true` to the
`physics` block of the JSON config.  The solver reads the pre-computed erosion tables
from `model_data/dust_sublimation/` at start-up:

```json
"physics": {
    "dust_accretion":    true,
    "dust_sputtering":   true,
    "dust_sublimation":  true,
    "dust_coagulation":  true,
    "pah_accretion":     false,
    "pah_photolysis":    false
}
```

The fractional dust mass-loss rate from sublimation is
$\dot{\rho}_\text{sub} / \rho = -3\,\langle\varepsilon\rangle$, where the
factor 3 converts the specific erosion rate ε = |da/dt|/a to a mass rate
(since $m \propto a^3$).  The liberated mass is returned to the gas phase
distributed over the elemental composition of each grain bin.


In [ ]:
import copy, tempfile, os, json as _json

# ---- Build a config with sublimation enabled (high G0 to trigger it) ----
subl_cfg = copy.deepcopy(cfg)
subl_cfg["environment"]["radiation_field_G0"] = 1e5   # intense UV
subl_cfg["environment"]["gas_temperature_K"]  = 1e4   # hot gas
subl_cfg["physics"]["dust_sublimation"] = True
subl_cfg["physics"]["dust_accretion"]   = False       # turn off competing process
subl_cfg["solver"]["t_end_Myr"]         = 0.01        # short run to see sublimation
subl_cfg["solver"]["type"]              = "rk4"

# Write to a temporary file and run
with tempfile.NamedTemporaryFile(
    mode="w", suffix=".json", delete=False
) as tmp:
    _json.dump(subl_cfg, tmp)
    tmp_path = tmp.name

from pycalima.solvers.run_chemistry import run_chemistry

result = run_chemistry(tmp_path, verbose=False)
os.unlink(tmp_path)

# run_chemistry returns: y_dust_init, y_dust_final, state, diagnostics, t_end_s
diag        = result["diagnostics"]
state_final = result["state"]
y_dust_0    = result["y_dust_init"]
y_dust_f    = result["y_dust_final"]
t_reached   = diag["tau"]           # [s] — actual final time reached

print(f"Steps accepted : {diag['naccepted']:d}  rejected: {diag['nrejected']:d}")
print(f"Time reached   : {t_reached / 3.1536e13:.4f} Myr  "
      f"(target {subl_cfg['solver']['t_end_Myr']:.4f} Myr)")
print()

# Compare initial vs final dust mass in each bin
for db in state_final.dust_bins:
    idx         = db.bin_index + state_final.npah
    frac_change = (y_dust_f[idx] - y_dust_0[idx]) / (y_dust_0[idx] + 1e-99)
    print(f"  {db.bin_id} ({db.composition:10s}, a={db.asize_micron:.3f} µm): "
          f"Δm/m = {frac_change * 100:+.2f}%")
